# Market Basket Analysis with Apriori (Week 4 tutorial, worked example)

*Adapted from Chris Moffitt, "Introduction to Market Basket Analysis in Python", Practical Business Python, 3 July 2017 (https://pbpython.com/market-basket-analysis.html). Data: the "Online Retail" dataset (D. Chen, UCI Machine Learning Repository): the invoices of a UK-based online gift shop from 1 December 2010 to 9 December 2011.*

**What this notebook covers:** which products are often bought together? Each invoice is one *transaction* (a "basket"). The notebook turns the invoices into a table with one row per basket and one True/False column per product, finds the *frequent itemsets* with the Apriori algorithm from the lecture, and turns them into *association rules* A → B with their support, confidence and lift. It does this for the customers in France, and then for the customers in Germany.

**How to run it:** this notebook runs in your browser, so there is nothing to install. Click a code cell and press **Shift + Enter** to run it and move to the next one, or use **Run → Run All Cells**. Run the cells in order, because later cells use the tables built earlier. The first code cell installs `mlxtend` into the browser, which takes a few seconds the first time. The data file `Online Retail (France and Germany).csv` must stay in the same folder as this notebook. It holds the 18,052 invoice lines from France and Germany; the full dataset has 541,909 lines from all countries.

**What to expect:** France has 392 baskets and 1,563 different products. With a minimum support of 7%, Apriori finds 51 frequent itemsets, which give 26 rules with a lift of at least 1. Eight of them have lift ≥ 6 and confidence ≥ 0.8: the green and red alarm clocks, and the red spotty paper cups, plates and napkins, are bought together far more often than chance would suggest. Germany (457 baskets, minimum support 5%) gives 3 strong rules (lift ≥ 4, confidence ≥ 0.5) for plaster tins and charlotte bags. The order of the rows in the rule tables can change from one run to the next; the rules and their numbers stay the same.

## 1. Load the libraries and the data

The first line installs `mlxtend`, the library that provides `apriori` and `association_rules`. The cell then imports pandas and reads the invoice lines into a DataFrame called `df`. Each row is one product line on an invoice: invoice number, stock code, product description, quantity, date, unit price, customer and country. `parse_dates` turns the text dates into real dates.

In [1]:
%pip install mlxtend
import pandas as pd
from mlxtend.frequent_patterns import apriori  # finds the frequent itemsets
from mlxtend.frequent_patterns import association_rules  # turns frequent itemsets into rules A -> B

# One row per invoice line (France and Germany only, see the opening cell)
df = pd.read_csv('Online Retail (France and Germany).csv', parse_dates=['InvoiceDate'], date_format='%m/%d/%Y %H:%M')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536370,22728,ALARM CLOCK BAKELIKE PINK,24,2010-12-01 08:45:00,3.75,12583.0,France
1,536370,22727,ALARM CLOCK BAKELIKE RED,24,2010-12-01 08:45:00,3.75,12583.0,France
2,536370,22726,ALARM CLOCK BAKELIKE GREEN,12,2010-12-01 08:45:00,3.75,12583.0,France
3,536370,21724,PANDA AND BUNNIES STICKER SHEET,12,2010-12-01 08:45:00,0.85,12583.0,France
4,536370,21883,STARS GIFT TAPE,24,2010-12-01 08:45:00,0.65,12583.0,France


## 2. Clean the data

There is a little cleanup we need to do. First, some of the descriptions have spaces that need to be removed. We’ll also drop the rows that don’t have invoice numbers and remove the credit transactions (those with invoice numbers containing C).

In [2]:
df['Description'] = df['Description'].str.strip()  # remove spaces at the start and end of product names
df.dropna(axis=0, subset=['InvoiceNo'], inplace=True)  # drop lines without an invoice number
df['InvoiceNo'] = df['InvoiceNo'].astype('str')
df = df[~df['InvoiceNo'].str.contains('C')]  # invoice numbers containing C are credit notes (returns)

## 3. Build one basket per invoice

After the cleanup, we need to consolidate the items into 1 transaction per row with each product 1 hot encoded. For the sake of keeping the data set small, I’m only looking at sales for France. However, in additional code below, I will compare these results to sales from Germany. Further country comparisons would be interesting to investigate.

The code keeps the French lines, adds up the quantity of each product on each invoice, and `unstack` spreads the products into columns. A product that is not on an invoice gets 0. `basket.head()` shows the first five baskets.

In [3]:
basket = (df[df['Country'] =="France"]
          .groupby(['InvoiceNo', 'Description'])['Quantity']  # total quantity of each product on each invoice
          .sum().unstack().reset_index().fillna(0)  # one column per product; 0 if not bought
          .set_index('InvoiceNo'))  # one row per invoice (basket)

In [4]:
basket.head() #check the new basket 

Description,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE SKULL,12 PENCILS TALL TUBE POSY,12 PENCILS TALL TUBE RED RETROSPOT,12 PENCILS TALL TUBE WOODLAND,...,WRAP VINTAGE PETALS DESIGN,YELLOW COAT RACK PARIS FASHION,YELLOW GIANT GARDEN THERMOMETER,YELLOW SHARK HELICOPTER,ZINC STAR T-LIGHT HOLDER,ZINC FOLKART SLEIGH BELLS,ZINC HERB GARDEN CONTAINER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536370,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536852,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536974,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
537065,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
537463,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. One-hot encode the baskets

There are a lot of zeros in the data but we also need to make sure any positive values are converted to a 1 and anything less than or equal to 0 is set to 0. This step will complete the one hot encoding of the data and remove the postage column (since that charge is not one we wish to explore):

`map` applies `encode_units` to every cell of the table. `astype(bool)` then stores the 1s and 0s as True and False, the data type that `apriori` expects.

In [5]:
def encode_units(x):
    if x <= 0:
        return 0  # not bought (or returned)
    if x >= 1:
        return 1  # bought at least once

basket_sets = basket.map(encode_units).astype(bool)  # 1/0 -> True/False
basket_sets.drop('POSTAGE', inplace=True, axis=1)  # postage is a charge, not a product

## 5. Find the frequent itemsets

Now that the data is structured properly, we can generate frequent item sets that have a support of at least 7% (this number was chosen so that I could get enough useful examples):

In [6]:
# keep the itemsets that appear in at least 7% of the French baskets
frequent_itemsets = apriori(basket_sets, min_support=0.07, use_colnames=True)

## 6. Generate the association rules

The final step is to generate the rules with their corresponding support, confidence and lift: For a rule A → B (see the lecture slides on association rules and lift):

- **support** = the share of baskets that contain both A and B;
- **confidence** = support(A ∪ B) / support(A): how often B is bought when A is bought;
- **lift** = confidence / support(B): how much more often A and B are bought together than if they were independent. Lift > 1 means a positive association.

`metric="lift", min_threshold=1` keeps only the rules with a lift of at least 1. `mlxtend` also reports other measures (representativity, leverage, conviction, Zhang's metric, Jaccard, certainty, Kulczynski) that we do not use here. The order of the rows can change from one run to the next.

In [7]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)  # all rules with lift >= 1
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({ALARM CLOCK BAKELIKE PINK}),frozenset({ALARM CLOCK BAKELIKE GREEN}),0.102041,0.096939,0.073980,0.725000,7.478947,1.0,0.064088,3.283859,0.964734,0.591837,0.695480,0.744079
1,frozenset({ALARM CLOCK BAKELIKE GREEN}),frozenset({ALARM CLOCK BAKELIKE PINK}),0.096939,0.102041,0.073980,0.763158,7.478947,1.0,0.064088,3.791383,0.959283,0.591837,0.736244,0.744079
2,frozenset({ALARM CLOCK BAKELIKE RED}),frozenset({ALARM CLOCK BAKELIKE GREEN}),0.094388,0.096939,0.079082,0.837838,8.642959,1.0,0.069932,5.568878,0.976465,0.704545,0.820431,0.826814
3,frozenset({ALARM CLOCK BAKELIKE GREEN}),frozenset({ALARM CLOCK BAKELIKE RED}),0.096939,0.094388,0.079082,0.815789,8.642959,1.0,0.069932,4.916181,0.979224,0.704545,0.796590,0.826814
4,frozenset({ALARM CLOCK BAKELIKE RED}),frozenset({ALARM CLOCK BAKELIKE PINK}),0.094388,0.102041,0.073980,0.783784,7.681081,1.0,0.064348,4.153061,0.960466,0.604167,0.759214,0.754392


Now, the tricky part is figuring out what this tells us. For instance, we can see that there are quite a few rules with a high lift value which means that it occurs more frequently than would be expected given the number of transaction and product combinations. We can also see several where the confidence is high as well. This part of the analysis is where the domain knowledge will come in handy. Since I do not have that, I’ll just look for a couple of illustrative examples.

We can filter the dataframe using standard pandas code. In this case, look for a large lift (6) and high confidence (.8):

In [8]:
# strong rules: lift of at least 6 and confidence of at least 0.8
rules[ (rules['lift'] >= 6) &
       (rules['confidence'] >= 0.8) ]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
2,frozenset({ALARM CLOCK BAKELIKE RED}),frozenset({ALARM CLOCK BAKELIKE GREEN}),0.094388,0.096939,0.079082,0.837838,8.642959,1.0,0.069932,5.568878,0.976465,0.704545,0.820431,0.826814
3,frozenset({ALARM CLOCK BAKELIKE GREEN}),frozenset({ALARM CLOCK BAKELIKE RED}),0.096939,0.094388,0.079082,0.815789,8.642959,1.0,0.069932,4.916181,0.979224,0.704545,0.796590,0.826814
16,frozenset({SET/6 RED SPOTTY PAPER PLATES}),frozenset({SET/20 RED RETROSPOT PAPER NAPKINS}),0.127551,0.132653,0.102041,0.800000,6.030769,1.0,0.085121,4.336735,0.956140,0.645161,0.769412,0.784615
18,frozenset({SET/6 RED SPOTTY PAPER PLATES}),frozenset({SET/6 RED SPOTTY PAPER CUPS}),0.127551,0.137755,0.122449,0.960000,6.968889,1.0,0.104878,21.556122,0.981725,0.857143,0.953609,0.924444
19,frozenset({SET/6 RED SPOTTY PAPER CUPS}),frozenset({SET/6 RED SPOTTY PAPER PLATES}),0.137755,0.127551,0.122449,0.888889,6.968889,1.0,0.104878,7.852041,0.993343,0.857143,0.872645,0.924444
20,"frozenset({SET/6 RED SPOTTY PAPER PLATES, SET/...",frozenset({SET/6 RED SPOTTY PAPER CUPS}),0.102041,0.137755,0.099490,0.975000,7.077778,1.0,0.085433,34.489796,0.956294,0.709091,0.971006,0.848611
21,"frozenset({SET/6 RED SPOTTY PAPER PLATES, SET/...",frozenset({SET/20 RED RETROSPOT PAPER NAPKINS}),0.122449,0.132653,0.099490,0.812500,6.125000,1.0,0.083247,4.625850,0.953488,0.639344,0.783824,0.781250
22,"frozenset({SET/20 RED RETROSPOT PAPER NAPKINS,...",frozenset({SET/6 RED SPOTTY PAPER PLATES}),0.102041,0.127551,0.099490,0.975000,7.644000,1.0,0.086474,34.897959,0.967949,0.764706,0.971345,0.877500


**What do you notice?** The rule *green alarm clock → red alarm clock* and the reverse rule *red → green* have the same lift (8.64) but different confidence (0.82 and 0.84). Why?

<details>
<summary>Show the answer</summary>

Lift is symmetric: lift(A → B) = support(A ∪ B) / (support(A) × support(B)), which gives the same value in both directions. Confidence divides only by the support of the left-hand side. The green clock is in slightly more baskets (9.7%) than the red one (9.4%), so confidence(green → red) = 0.0791 / 0.0969 = 0.82 is a little lower than confidence(red → green) = 0.0791 / 0.0944 = 0.84.
</details>

In looking at the rules, it seems that the green and red alarm clocks are purchased together and the red paper cups, napkins and plates are purchased together in a manner that is higher than the overall probability would suggest.

At this point, you may want to look at how much opportunity there is to use the popularity of one product to drive sales of another. For instance, we can see that we sell 340 Green Alarm clocks but only 316 Red Alarm Clocks in France so maybe we can drive more Red Alarm Clock sales through recommendations?

The next two cells add up the quantity of each clock over all French baskets.

In [10]:
basket['ALARM CLOCK BAKELIKE GREEN'].sum()

np.float64(340.0)

In [11]:
basket['ALARM CLOCK BAKELIKE RED'].sum()

np.float64(316.0)

## 7. Compare with Germany

What is also interesting is to see how the combinations vary by country of purchase. Let’s check out what some popular combinations might be in Germany:

The cell repeats steps 3 to 6 for the German invoices, with a lower minimum support (5%) and lower thresholds for the strong rules (lift ≥ 4, confidence ≥ 0.5).

In [12]:
basket2 = (df[df['Country'] =="Germany"]
          .groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))  # same steps as for France

basket_sets2 = basket2.map(encode_units).astype(bool)
basket_sets2.drop('POSTAGE', inplace=True, axis=1)
frequent_itemsets2 = apriori(basket_sets2, min_support=0.05, use_colnames=True)  # at least 5% of the baskets
rules2 = association_rules(frequent_itemsets2, metric="lift", min_threshold=1)

# strong rules: lift of at least 4 and confidence of at least 0.5
rules2[ (rules2['lift'] >= 4) &
        (rules2['confidence'] >= 0.5)]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({PLASTERS IN TIN CIRCUS PARADE}),frozenset({PLASTERS IN TIN WOODLAND ANIMALS}),0.115974,0.137856,0.067834,0.584906,4.242887,1.0,0.051846,2.076984,0.864580,0.364706,0.518533,0.538485
6,frozenset({PLASTERS IN TIN SPACEBOY}),frozenset({PLASTERS IN TIN WOODLAND ANIMALS}),0.107221,0.137856,0.061269,0.571429,4.145125,1.0,0.046488,2.011670,0.849877,0.333333,0.502901,0.507937
10,frozenset({RED RETROSPOT CHARLOTTE BAG}),frozenset({WOODLAND CHARLOTTE BAG}),0.070022,0.126915,0.059081,0.843750,6.648168,1.0,0.050194,5.587746,0.913551,0.428571,0.821037,0.654634


It seems that German customers often buy the Circus Parade or Spaceboy plaster tins together with the Woodland Animals tin, and the Red Retrospot charlotte bag together with the Woodland one.

In all seriousness, an analyst that has familiarity with the data would probably have a dozen different questions that this type of analysis could drive. I did not replicate this analysis for additional countries or customer combos but the overall process would be relatively simple given the basic pandas code shown above.

**Try it:** in section 5, change `min_support=0.07` to `min_support=0.05` and run the cells of sections 5 and 6 again. What happens to the number of frequent itemsets (`len(frequent_itemsets)`) and rules (`len(rules)`)?

<details>
<summary>Show the answer</summary>

You get 113 frequent itemsets instead of 51, and 86 rules instead of 26 (15 of them pass the lift ≥ 6 and confidence ≥ 0.8 filter instead of 8). A lower threshold keeps many more itemsets, and their number grows quickly: this is the "computational complexity" point from the lecture, and why the choice of minimum support matters.
</details>